# VER — Inclusions Simples Sphériques (Mori-Tanaka)

Ce notebook étudie les propriétés élastiques effectives d'un milieu composite constitué
d'une **matrice isotrope** renforcée par des **inclusions simples** (sphériques ou sphéroïdales),
à l'aide du **schéma de Mori-Tanaka** (bibliothèque `echoes`).

Trois paramètres sont explorés successivement :
1. **Fraction volumique** $f_i$ des inclusions
2. **Contraste élastique** $E_i / E_m$
3. **Forme** des inclusions (rapport d'aspect $\omega = a/c$)


In [ ]:
#! /usr/bin/env python
# ─── Imports et configuration globale ────────────────────────────────────────
import numpy as np
from echoes import *
import matplotlib.pyplot as plt

# Paramètres graphiques globaux : axes épais, labels en gras
plt.rcParams.update({
    "axes.linewidth":  2,
    "axes.labelsize":  16,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.major.width": 1.5,
    "ytick.major.width": 1.5
})


## Fonctions utilitaires

### `Chom_Simple` — inclusion sphérique à une seule phase

Crée un VER avec une matrice + une inclusion sphérique (ou sphéroïdale si `omega` est fourni)
et retourne le tenseur homogénéisé via Mori-Tanaka.

**Paramètres :**
- `Em` : Module de Young de la matrice (GPa)
- `Ei` : Module de Young de l'inclusion (GPa)
- `fi` : Fraction volumique de l'inclusion
- `omega` *(optionnel)* : rapport d'aspect sphéroïdal ($\omega < 1$ oblate, $\omega > 1$ prolate)


In [ ]:
def Chom_Simple(Em, Ei, fi, omega=None):
    """
    Homogénéisation Mori-Tanaka d'un VER matrice + inclusion simple.

    Si omega est None  → inclusion sphérique.
    Si omega est fourni → inclusion sphéroïdale de rapport d'aspect omega.
    """
    Ci   = stiff_Enu(Ei, 0.2)     # Tenseur de rigidité de l'inclusion (nu=0.2)
    Cmat = stiff_Enu(Em, 0.2)     # Tenseur de rigidité de la matrice

    ver = rve(matrix="MATRIX")
    ver["MATRIX"] = ellipsoid(shape=spherical, prop={"C": Cmat}, fraction=1. - fi)

    if omega is None:
        # Inclusion sphérique multi-couches (1 couche = sphère simple)
        ver["INCL"] = sphere_nlayers(radii=[2], prop={"C": [Ci]}, fraction=fi)
    else:
        # Inclusion sphéroïdale : rapport d'aspect variable
        ver["INCL"] = ellipsoid(shape=spheroidal(aspect_ratio=omega), prop={"C": Ci}, fraction=fi)

    Chom = homogenize(prop="C", rve=ver, scheme=MT, verbose=False)
    return Chom


---
## 1. Effet de la fraction volumique $f_i$

On fait varier $f_i \in [0, 0.6]$ pour quatre contrastes $E_i/E_m \in \{0.5, 1, 2, 5\}$.

- **Matrice** : $E_m = 10$ GPa
- **Inclusions** : $E_i \in \{5, 10, 20, 50\}$ GPa


In [ ]:
# ─── Paramètres ───────────────────────────────────────────────────────────────
fi_values = np.linspace(0, 0.6, 100)      # Fraction volumique : 0 à 60 %
Em = 10                                    # Module matrice (GPa)
E1 = [5, 10, 20, 50]                       # Modules d'inclusion (GPa)

# Labels, couleurs et marqueurs par courbe
ratios    = [f"$E_i/E_m = {e/Em:.1g}$" for e in E1]
couleurs  = ['blue', 'green', 'black', 'red']
marqueurs = ['o', 's', '^', 'D']

# Tableaux de stockage des résultats (4 contrastes × 100 points)
result_E  = np.zeros((4, len(fi_values)))
result_K  = np.zeros((4, len(fi_values)))
result_mu = np.zeros((4, len(fi_values)))

# ─── Calcul ───────────────────────────────────────────────────────────────────
for i, Ei in enumerate(E1):
    for j, fi in enumerate(fi_values):
        Chom = Chom_Simple(Em, Ei, fi)
        result_E[i, j]  = Chom.E
        result_K[i, j]  = Chom.k
        result_mu[i, j] = Chom.mu

# ─── Figure 1 : Module de Young effectif ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for i in range(len(E1)):
    ax.plot(fi_values, result_E[i, :], f'{marqueurs[i]}-', color=couleurs[i],
            label=ratios[i], markevery=5)
ax.set_xlabel("Fraction volumique de l'inclusion $f_i$")
ax.set_ylabel("Module de Young effectif $\\tilde{E}$ (GPa)")
ax.legend(loc='best')
ax.set_xlim([0, 0.6])
ax.grid(True)
plt.tight_layout()
plt.savefig('module_young_effectif_mori_tanaka.png', dpi=150)
plt.show()

# ─── Figure 2 : Module de compressibilité effectif ───────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for i in range(len(E1)):
    ax.plot(fi_values, result_K[i, :], f'{marqueurs[i]}-', color=couleurs[i],
            label=ratios[i], markevery=5)
ax.set_xlabel("Fraction volumique de l'inclusion $f_i$")
ax.set_ylabel("Module de compressibilité effectif $\\tilde{K}$ (GPa)")
ax.legend(loc='best')
ax.set_xlim([0, 0.6])
ax.grid(True)
plt.tight_layout()
plt.savefig('module_compressibilite_effectif_mori_tanaka.png', dpi=150)
plt.show()

# ─── Figure 3 : Module de cisaillement effectif ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for i in range(len(E1)):
    ax.plot(fi_values, result_mu[i, :], f'{marqueurs[i]}-', color=couleurs[i],
            label=ratios[i], markevery=5)
ax.set_xlabel("Fraction volumique de l'inclusion $f_i$")
ax.set_ylabel("Module de cisaillement effectif $\\tilde{\\mu}$ (GPa)")
ax.legend(loc='best')
ax.set_xlim([0, 0.6])
ax.grid(True)
plt.tight_layout()
plt.savefig('module_cisaillement_effectif_mori_tanaka.png', dpi=150)
plt.show()


---
## 2. Effet du contraste élastique $E_i / E_m$

On fait varier $E_i \in [1, 50]$ GPa (contraste de 0.1 à 5) pour quatre fractions volumiques
$f_i \in \{0.3, 0.4, 0.5, 0.6\}$.

- **Matrice** : $E_m = 10$ GPa
- Plus la fraction volumique est grande, plus l'effet du contraste est amplifié.


In [ ]:
# ─── Paramètres ───────────────────────────────────────────────────────────────
Ei_values = np.linspace(1, 50, 140)          # Module inclusion : 1 à 50 GPa
Em        = 10                               # Module matrice (GPa)
fi_values = [0.3, 0.4, 0.5, 0.6]            # Fractions volumiques testées
couleurs  = ['blue', 'green', 'black', 'red']
marqueurs = ['o', 's', '^', 'D']

# Stockage : dictionnaire fi → {E, K, mu, contraste}
resultats = {fi: {'E': [], 'K': [], 'mu': [], 'contraste': []} for fi in fi_values}

# ─── Calcul ───────────────────────────────────────────────────────────────────
for fi in fi_values:
    for Ei in Ei_values:
        Chom = Chom_Simple(Em, Ei, fi)
        resultats[fi]['E'].append(Chom.E)
        resultats[fi]['K'].append(Chom.k)
        resultats[fi]['mu'].append(Chom.mu)
        resultats[fi]['contraste'].append(Ei / Em)   # rapport adimensionnel

# ─── Tracé des 3 modules en fonction du contraste ────────────────────────────
proprietes = [
    ('E',  "Contraste de module $E_i/E_m$", "Module de Young effectif $\\tilde{E}$ (GPa)",         'effet_contraste_E.png'),
    ('K',  "Contraste de module $E_i/E_m$", "Module de compressibilité effectif $\\tilde{k}$ (GPa)", 'effet_contraste_K.png'),
    ('mu', "Contraste de module $E_i/E_m$", "Module de cisaillement effectif $\\tilde{\\mu}$ (GPa)", 'effet_contraste_mu.png'),
]

for prop, xlabel, ylabel, nom_fichier in proprietes:
    fig, ax = plt.subplots(figsize=(9, 6))
    for fi, couleur, marqueur in zip(fi_values, couleurs, marqueurs):
        ax.plot(resultats[fi]['contraste'], resultats[fi][prop],
                marker=marqueur, linestyle='-', color=couleur,
                label=f"$f_i = {fi}$", markevery=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(loc='best')
    ax.grid(True)
    ax.set_xlim([min(resultats[fi_values[0]]['contraste']),
                 max(resultats[fi_values[0]]['contraste'])])
    plt.tight_layout()
    plt.savefig(nom_fichier, dpi=150, bbox_inches='tight')
    plt.show()


---
## 3. Effet de la forme de l'inclusion — cas contrasté ($E_i = 70$ GPa)

Le **rapport d'aspect** $\omega = a/c$ caractérise la forme de l'inclusion sphéroïdale :
- $\omega < 1$ : oblate (disque aplati)
- $\omega = 1$ : sphère
- $\omega > 1$ : prolate (fibre élancée)

On balaye $\omega \in [0.1, 10]$ (échelle logarithmique) pour $E_i = 70$ GPa ($E_i/E_m = 7$).

> **Résultat remarquable :** la sphère ($\omega = 1$) est la géométrie la **moins efficace** ;
> tout écart vers oblate ou prolate améliore les propriétés effectives.


In [ ]:
# ─── Paramètres ───────────────────────────────────────────────────────────────
omega_values = np.logspace(-1, 1, 100)     # Rapport d'aspect : 0.1 à 10 (log)
Em = 10
Ei = 70                                    # Fort contraste E_i/E_m = 7
fi_values = [0.3, 0.4, 0.5, 0.6]
couleurs  = ['blue', 'green', 'black', 'red']
marqueurs = ['o', 's', '^', 'D']

# Stockage
resultats = {fi: {'E': [], 'K': [], 'mu': []} for fi in fi_values}

# ─── Calcul ───────────────────────────────────────────────────────────────────
for fi in fi_values:
    for omega in omega_values:
        Chom = Chom_Simple(Em, Ei, fi, omega=omega)
        resultats[fi]['E'].append(Chom.E)
        resultats[fi]['K'].append(Chom.k)
        resultats[fi]['mu'].append(Chom.mu)

# Impression des valeurs minimales (minimum atteint pour la sphère)
print("Valeurs minimales (E_i=70 GPa) :")
print(f"  min K  (fi=0.3) = {min(resultats[0.3]['K']):.4f} GPa")
print(f"  min mu (fi=0.3) = {min(resultats[0.3]['mu']):.4f} GPa")
print(f"  min E  (fi=0.5) = {min(resultats[0.5]['E']):.4f} GPa")
print(f"  min K  (fi=0.5) = {min(resultats[0.5]['K']):.4f} GPa")
print(f"  min mu (fi=0.5) = {min(resultats[0.5]['mu']):.4f} GPa")

# ─── Tracé ────────────────────────────────────────────────────────────────────
proprietes = [
    ('E',  "Module de Young effectif $\\tilde{E}$ (GPa)",           'effet_forme_E.png'),
    ('K',  "Module de compressibilité effectif $\\tilde{k}$ (GPa)",  'effet_forme_K.png'),
    ('mu', "Module de cisaillement effectif $\\tilde{\\mu}$ (GPa)", 'effet_forme_mu.png'),
]

for prop, ylabel, nom_fichier in proprietes:
    fig, ax = plt.subplots(figsize=(9, 6))
    for fi, couleur, marqueur in zip(fi_values, couleurs, marqueurs):
        ax.semilogx(omega_values, resultats[fi][prop],
                    marker=marqueur, linestyle='-', color=couleur,
                    label=f"$f_i = {fi}$", markevery=10)

    # Ligne verticale : sphère (omega = 1)
    ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2,
               alpha=0.8, label='Sphère ($\\omega = 1$)')

    # Annotations des zones oblate / sphère / prolate
    y_max = ax.get_ylim()[1]
    ax.text(0.3, y_max * 0.97, 'Oblate\n(disque)', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=1))
    ax.text(1.0, y_max * 0.70, 'Sphère', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightgray', alpha=1))
    ax.text(5,   y_max * 0.97, 'Prolate\n(fibre)', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=1))

    ax.set_xlabel("Rapport d'aspect $\\omega = a/c$")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=10, loc='best', framealpha=0.95)
    ax.grid(True)
    ax.set_xlim([omega_values.min(), omega_values.max()])
    plt.tight_layout()
    plt.savefig(nom_fichier)
    plt.show()


---
## 4. Effet de la forme — cas neutre ($E_i = E_m = 10$ GPa)

Même analyse que la section 3, mais avec des inclusions de **même rigidité que la matrice**.

> **Résultat attendu :** les propriétés effectives ne dépendent quasiment pas de la forme
> lorsque le contraste est nul — validation de la cohérence du modèle.


In [ ]:
# ─── Paramètres (inclusions identiques à la matrice) ─────────────────────────
omega_values = np.logspace(-1, 1, 100)
Em = 10
Ei = 10                                    # Pas de contraste : E_i = E_m
fi_values = [0.3, 0.4, 0.5, 0.6]
couleurs   = ['blue', 'green', 'black', 'red']
marqueurs  = ['o', 's', '^', 'D']
markevery  = [6, 8, 11, 15]               # Espacement différent par courbe
ombres     = [False, True, False, True]   # Effet d'ombre sur certaines courbes

# Stockage
resultats = {fi: {'E': [], 'K': [], 'mu': []} for fi in fi_values}

# ─── Calcul ───────────────────────────────────────────────────────────────────
for fi in fi_values:
    for omega in omega_values:
        Chom = Chom_Simple(Em, Ei, fi, omega=omega)
        resultats[fi]['E'].append(Chom.E)
        resultats[fi]['K'].append(Chom.k)
        resultats[fi]['mu'].append(Chom.mu)

# ─── Tracé ────────────────────────────────────────────────────────────────────
proprietes = [
    ('E',  "Module de Young effectif $\\tilde{E}$ (GPa)",           'effet_forme_E_id.png'),
    ('K',  "Module de compressibilité effectif $\\tilde{k}$ (GPa)",  'effet_forme_K_id.png'),
    ('mu', "Module de cisaillement effectif $\\tilde{\\mu}$ (GPa)", 'effet_forme_mu_id.png'),
]

for prop, ylabel, nom_fichier in proprietes:
    fig, ax = plt.subplots(figsize=(9, 6))

    for fi, couleur, marqueur, me, ombre in zip(fi_values, couleurs, marqueurs, markevery, ombres):
        valeurs = resultats[fi][prop]

        if ombre:
            # Halo gris semi-transparent pour mettre en valeur la courbe
            ax.semilogx(omega_values, valeurs,
                        linestyle='-', color='gray', linewidth=4,
                        alpha=0.35, zorder=1)

        # Courbe principale
        ax.semilogx(omega_values, valeurs,
                    marker=marqueur, linestyle='-', color=couleur,
                    label=f"$f_i = {fi}$", markevery=me,
                    linewidth=1.8, zorder=2)

    # Ligne verticale : sphère
    ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2,
               alpha=0.8, label='Sphère ($\\omega = 1$)')

    # Annotations
    y_max = ax.get_ylim()[1]
    ax.text(0.3, y_max * 0.97, 'Oblate\n(disque)', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=1))
    ax.text(1.0, y_max * 0.97, 'Sphère', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightgray', alpha=1))
    ax.text(5,   y_max * 0.97, 'Prolate\n(fibre)', ha='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=1))

    ax.set_xlabel("Rapport d'aspect $\\omega = a/c$")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=10, loc='best', framealpha=0.95)
    ax.grid(True)
    ax.set_xlim([omega_values.min(), omega_values.max()])
    plt.tight_layout()
    plt.tick_params(axis='both', labelsize=14)
    plt.savefig(nom_fichier)
    plt.show()
